In [ ]:
from pyspark.sql.functions import (
    col,
    explode,
    lit,
    max,
    min,
    quarter,
    row_number,
    sequence,
    year,
)
from pyspark.sql.window import Window


sales = spark.read.table("silver_salesorderheader")
customers = spark.read.table("silver_customer_scd2")

customer_window = Window.orderBy("customer_id", "valid_from")

dim_customer = (
    customers
    .withColumn(
        "customer_sk",
        row_number().over(customer_window),
    )
    .select(
        "customer_sk",
        "customer_id",
        "customer_name",
        "customer_type",
        "territory_id",
        "valid_from",
        "valid_to",
        "is_current",
        "source_system",
    )
)

date_range = sales.select(
    min("order_date").alias("first_date"),
    max("order_date").alias("last_date"),
).first()

dim_date = (
    spark.range(1)
    .select(
        explode(
            sequence(
                lit(date_range["first_date"]),
                lit(date_range["last_date"]),
            )
        ).alias("date")
    )
    .select(
        col("date").alias("date_key"),
        year("date").alias("year"),
        quarter("date").alias("quarter"),
    )
)

dim_territory = (
    sales
    .select("territory_id")
    .where(col("territory_id").isNotNull())
    .distinct()
    .withColumn(
        "territory_sk",
        row_number().over(Window.orderBy("territory_id")),
    )
    .select("territory_sk", "territory_id")
)

fact_sales = (
    sales.alias("sales")
    .join(
        dim_customer.alias("customer"),
        (col("sales.customer_id") == col("customer.customer_id"))
        & (
            col("sales.order_date").between(
                col("customer.valid_from"),
                col("customer.valid_to"),
            )
        ),
        "left",
    )
    .select(
        col("sales.sales_order_id"),
        col("sales.order_date").alias("date_key"),
        col("customer.customer_sk"),
        col("sales.territory_id"),
        col("sales.is_online_order"),
        col("sales.subtotal_amount"),
        col("sales.tax_amount"),
        col("sales.freight_amount"),
        col("sales.load_date"),
        col("sales.source_system"),
        col("sales.silver_processed_timestamp"),
    )
)

for table, name in [
    (dim_customer, "dim_customer"),
    (dim_date, "dim_date"),
    (dim_territory, "dim_territory"),
    (fact_sales, "fact_sales"),
]:
    (
        table.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(name)
    )

print(f"dim_customer : {dim_customer.count()}")
print(f"dim_date : {dim_date.count()}")
print(f"dim_territory : {dim_territory.count()}")
print(f"fact_sales : {fact_sales.count()}")

In [ ]:
missing_customer_keys = fact_sales.filter(
    col("customer_sk").isNull()
).count()

duplicate_sales = (
    fact_sales
    .groupBy("sales_order_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

missing_date_keys = fact_sales.filter(
    col("date_key").isNull()
).count()

print(f"Faits sans customer_sk : {missing_customer_keys}")
print(f"Faits dupliqués : {duplicate_sales}")
print(f"Faits sans date_key : {missing_date_keys}")

if missing_customer_keys > 0:
    raise ValueError("Des faits n'ont pas de customer_sk.")

if duplicate_sales > 0:
    raise ValueError("Des SalesOrderID sont dupliqués.")

if missing_date_keys > 0:
    raise ValueError("Des faits n'ont pas de date_key.")

print("Contrôles Gold Fabric réussis.")

In [ ]:
from pyspark.sql.functions import sum as sum_, count

reconciliation = fact_sales.select(
    count("*").alias("orders"),
    sum_("subtotal_amount").alias("subtotal"),
    sum_("tax_amount").alias("tax"),
    sum_("freight_amount").alias("freight"),
)

display(reconciliation)

In [ ]:
from pyspark.sql.functions import col, sum as sum_, count

display(
    fact_sales
    .groupBy("load_date")
    .agg(
        count("*").alias("orders"),
        sum_("subtotal_amount").alias("subtotal"),
        sum_("tax_amount").alias("tax"),
        sum_("freight_amount").alias("freight"),
    )
    .orderBy("load_date")
)